In [1]:
# ######### used this part for fixing problems running on ARC #

import os


os.environ['HF_HOME'] = '/projects/abbott_lab/Users/ishtiaque/hfmodels/'
os.environ['HF_HUB_CACHE'] = '/projects/abbott_lab/Users/ishtiaque/hfmodels/'
os.environ['XDG_CACHE_HOME'] = '/projects/abbott_lab/Users/ishtiaque/hfmodels/'
os.environ['NB_USER'] = 'ishtiahmed'#'ishtiaqueahmedk'
os.environ['TRANSFORMERS_CACHE'] = '/projects/abbott_lab/Users/ishtiaque/hfmodels/'
os.environ['HF_DATASETS_CACHE'] = '/projects/abbott_lab/Users/ishtiaque/hfmodels/'




In [2]:
import torch
import os 
import json
import random
from tqdm import tqdm
from collections import Counter 

# for qwen2.5-vl
from transformers import Qwen2_5_VLForConditionalGeneration, AutoTokenizer, AutoProcessor
from qwen_vl_utils import process_vision_info 




/projects/abbott_lab/Users/ishtiaque/env/qwen_2_5_VL_a100/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/projects/abbott_lab/Users/ishtiaque/env/qwen_2_5_VL_a100/lib/python3.10/site-packages/transformers/utils/hub.py:107: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(
/projects/abbott_lab/Users/ishtiaque/env/qwen_2_5_VL_a100/lib/python3.10/site-packages/torchvision/io/image.py:13: UserWarning: Failed to load image Python extension: '/projects/abbott_lab/Users/ishtiaque/env/qwen_2_5_VL_a100/lib/python3.10/site-packages/torchvision/image.so: undefined symbol: _ZN3c1017RegisterOperatorsD1Ev'If you don't plan on using image functionality from `torchvision.io`, you can ignore this warning. Otherwise, there might be 

In [3]:
# default: Load the model on the available device(s) 
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    "Qwen/Qwen2.5-VL-7B-Instruct", torch_dtype="auto", device_map="auto"
)
processor = AutoProcessor.from_pretrained("Qwen/Qwen2.5-VL-7B-Instruct")


`torch_dtype` is deprecated! Use `dtype` instead!
Loading checkpoint shards: 100%|██████████| 5/5 [00:02<00:00,  1.79it/s]
The image processor of type `Qwen2VLImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. Note that this behavior will be extended to all models in a future release.


In [4]:
def get_bird_images(images_folder):
# Path to the folder containing bird subfolders
    # images_folder = '/projects/abbott_lab/Users/ishtiaque/datasets/CUB_200_2011/images'
    
    # Dictionary to store bird names and their corresponding image paths
    bird_images = {}
    
    # Iterate over each subfolder in the images folder
    for folder in os.listdir(images_folder):
        bird_name = folder.split(".")[-1]  # Extract bird name from folder name
        folder_path = os.path.join(images_folder, folder)  # Path to the bird's folder
        
        # Initialize an empty list to store image paths for the current bird
        image_paths = []
        
        # Iterate over the image files in the bird's folder
        for image_file in os.listdir(folder_path):
            image_path = os.path.join(folder_path, image_file)  # Full path to the image file
            image_paths.append(image_path)  # Store the image path
        
        # Store the list of image paths in the dictionary under the bird's name
        bird_images[bird_name] = image_paths
    
    # Now bird_images contains a dictionary where the keys are bird names and the values are lists of image paths
    print(len(bird_images))
    return bird_images



In [5]:
def get_json_data(json_file_name):

    # negated_questions, modified_new_cub_class_descriptions_full_fake, #modified_new_cub_class_descriptionsx #modified_mcqs_description_only2, modified_mcqs_description_only

    # For Task 1 type 1: 
    
    with open(json_file_name, "r") as file: 
        json_data = json.load(file)
    print(len(json_data))
    return json_data






In [6]:
def get_medium_hard_data(json_data):
    
# Use this if the json file contain the medium and hard categories
    medium_data = []
    hard_data = []
    for data in json_data:
        if data['difficulty'] == "Medium":
            medium_data.append(data)
        else:
            hard_data.append(data)
    print(len(medium_data))
    print(len(hard_data))
    return medium_data, hard_data
    

In [7]:
def get_answer(image_path, query):
    messages = [
        {
            "role": "user",
            "content": [
                {
                    "type": "image",
                    "image": image_path,
                },
                {"type": "text", "text": query},
            ],
        }
    ]

    # Preparation for inference
    text = processor.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    image_inputs, video_inputs = process_vision_info(messages)
    inputs = processor(
        text=[text],
        images=image_inputs,
        # videos=video_inputs,
        padding=True,
        return_tensors="pt",
    )
    inputs = inputs.to("cuda")

    # Inference: Generation of the output
    generated_ids = model.generate(**inputs, max_new_tokens=128) 
    # print(generated_ids)

    generated_ids_trimmed = [
        out_ids[len(in_ids) :] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
    ]
    output_text = processor.batch_decode(
        generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
    )[0]
    
    return output_text

In [11]:
def run_eval(data_partition):
    
    # Counters for distribution
    true_distribution = Counter()
    predicted_distribution = Counter()
    
    results = []
    
    for i, item in tqdm(enumerate(data_partition)): # for easy part json_data, for medium_data, for hard_data 
        mcq_id = item['mcq_id']
        question = item['question']
        options = item['options']
        correct_answer = item['correct_answer']
    
        if mcq_id not in bird_images or not bird_images[mcq_id]:
            print(f"No image for {mcq_id}") 
            continue
    
        image_paths = bird_images[mcq_id][:5]
    
        # Format the prompt
        # formatted_prompt = f"{question}\n"

        if ("task_1a" in json_file_name): #correct ClassName
            formatted_prompt = f"{question} Ignore the descriptions and focus only on the class names.\n"
        elif ("task_1b" in json_file_name): #correct Description
            formatted_prompt = f"{question} Ignore the class names and focus only on the descriptions.\n"
        else:
            print(f"Error in File Name: {json_file_name}")
            print(asd)

        # formatted_prompt = f"{question} Ignore the class names and focus on the descriptions.\n" # 
        for k in ['A', 'B', 'C', 'D']:  # ['D', 'C', 'B', 'A'] for position bias checking ['A', 'B', 'C', 'D']
            formatted_prompt += f"{k}. {options[k]}\n"
    
        # Final prompt
        prompt = f""" Your answer or response must ONLY be a single index ('A', 'B', 'C', 'D'). Do not response with any other text. 
    
        {formatted_prompt}
    
        Answer: ('A', 'B', 'C', 'D')"""
    
        # Run the model
        for image_path in image_paths:
            model_output = get_answer(image_path, prompt)
            # print("Model Output: ", model_output)
    
            # Extract predicted answer (basic string search, can refine)
            predicted_answer = None
            for option in ['A', 'B', 'C', 'D']:
                if f"{option}" in model_output or f"{option}." in model_output:
                    predicted_answer = option
    
            # Update counters
            true_distribution[correct_answer] += 1
            if predicted_answer:
                predicted_distribution[predicted_answer] += 1
            
            results.append({
                'mcq_id': mcq_id,
                'image_path': image_path,
                'prompt': prompt,
                'model_output': model_output,
                'predicted_answer': predicted_answer,
                'correct_answer': correct_answer,
                'is_correct': predicted_answer == correct_answer
            })
    
    print(f"Results for file: {json_file_name}")
    # Accuracy summary 
    correct = sum(r['is_correct'] for r in results if r['predicted_answer'] is not None)
    total = len(results)
    print(f"Accuracy: {correct}/{total} = {correct / total:.2%}") 
    
    # Print distributions
    print("True Option Distribution:", dict(true_distribution))
    print("Predicted Option Distribution:", dict(predicted_distribution)) 

In [12]:

# json_files_list = ['negated_questions', 'new_cub_class_descriptions','new_cub_with_class_descriptions', 'modified_new_cub_class_descriptions_full_fake', 'modified_new_cub_class_descriptions_partial_fake', 'incorrect_class_correct_description', 'correct_class_incorrect_description']

bird_folder = "/projects/abbott_lab/Users/ishtiaque/datasets/CUB_200_2011/images"
food_folder = "/projects/abbott_lab/Users/ishtiaque/datasets/food-101/food-101/images"
aircraft_folder = "/projects/abbott_lab/Users/ishtiaque/datasets/fgvc-aircraft-2013b/data/test"
dogs_folder = "/projects/abbott_lab/Users/ishtiaque/datasets/stanford-dogs/images/Images"
car_folder = "/projects/abbott_lab/Users/ishtiaque/datasets/stanford-cars/train"

bird_list = [
    "new_cub_class_descriptions_task_1b",
    "new_cub_class_descriptions_task_1a",
]

food_list = [
    "new_food_class_descriptions_task_1a",
    "new_food_class_descriptions_task_1b",
]

aircraft_list = [
    "new_aircraft_class_descriptions_task_1a",
    "new_aircraft_class_descriptions_task_1b",
]

dogs_list = [
    "new_dogs_class_descriptions_task_1a",
    "new_dogs_class_descriptions_task_1b",
]

car_list = [
    "new_car_class_descriptions_task_1a",
    "new_car_class_descriptions_task_1b",
]

# json_files_list = food_list  #################### change!!!!!!!!!!!!!!!!!!!!
all_lists = [bird_list, food_list, aircraft_list, dogs_list, car_list]
folder_lists = [bird_folder, food_folder, aircraft_folder, dogs_folder, car_folder]

# all_lists = [food_list, aircraft_list, dogs_list, car_list]
# folder_lists = [food_folder, aircraft_folder, dogs_folder, car_folder]


for json_files_list, images_folder in zip(all_lists, folder_lists):

    bird_images = get_bird_images(images_folder)
    
    for json_file_name in json_files_list:
        
        json_file_name = f"/home/ishtiaqueahmedk/Research/VLM/fine_grained/mcq_json files/{json_file_name}.json"
        json_data = get_json_data(json_file_name)
    
        json_data[0]
        # json_data[1]
    
        medium_data, hard_data = get_medium_hard_data(json_data)
        
        print("\n----Medium----")
        run_eval(medium_data)
        print("\n----Hard----")
        run_eval(hard_data)
    

200
400
200
200

----Medium----


200it [02:15,  1.48it/s]


Results for file: /home/ishtiaqueahmedk/Research/VLM/fine_grained/mcq_json files/new_cub_class_descriptions_task_1b.json
Accuracy: 944/1000 = 94.40%
True Option Distribution: {'D': 245, 'C': 220, 'B': 235, 'A': 300}
Predicted Option Distribution: {'D': 243, 'C': 224, 'B': 246, 'A': 287}

----Hard----


200it [02:15,  1.47it/s]


Results for file: /home/ishtiaqueahmedk/Research/VLM/fine_grained/mcq_json files/new_cub_class_descriptions_task_1b.json
Accuracy: 222/1000 = 22.20%
True Option Distribution: {'D': 275, 'A': 260, 'B': 215, 'C': 250}
Predicted Option Distribution: {'D': 166, 'B': 427, 'C': 240, 'A': 167}
400
200
200

----Medium----


...

Results for file: /home/ishtiaqueahmedk/Research/VLM/fine_grained/mcq_json files/new_car_class_descriptions_task_1a.json
Accuracy: 610/980 = 62.24%
True Option Distribution: {'A': 245, 'D': 250, 'B': 245, 'C': 240}
Predicted Option Distribution: {'B': 331, 'A': 176, 'D': 224, 'C': 249}
392
196
196

----Medium----


196it [03:31,  1.08s/it]


Results for file: /home/ishtiaqueahmedk/Research/VLM/fine_grained/mcq_json files/new_car_class_descriptions_task_1b.json
Accuracy: 921/980 = 93.98%
True Option Distribution: {'B': 250, 'C': 300, 'D': 250, 'A': 180}
Predicted Option Distribution: {'B': 255, 'C': 304, 'A': 180, 'D': 241}

----Hard----


196it [03:31,  1.08s/it]

Results for file: /home/ishtiaqueahmedk/Research/VLM/fine_grained/mcq_json files/new_car_class_descriptions_task_1b.json
Accuracy: 361/980 = 36.84%
True Option Distribution: {'A': 245, 'D': 250, 'B': 245, 'C': 240}
Predicted Option Distribution: {'B': 388, 'A': 155, 'D': 161, 'C': 276}


# Cub-only